In [4]:
# Sel 1: Import Library
import os
import shutil
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [5]:
# Sel 2: Konfigurasi Direktori dan Parameter
RAW_DIR = '../data/raw/'
PROCESSED_DIR = '../data/processed/'
SPLIT_DIR = '../data/split/'          
CLASSES = ['healthy', 'sick']
IMG_SIZE = 224                         # Sesuai kebutuhan EfficientNet-B3
TEST_SIZE = 0.2
RANDOM_STATE = 42

In [6]:
# Sel 3: Split nama file per kelas, lalu salin fisik ke data/split/
def list_image_files(class_dir):
    valid_ext = ('.png', '.jpg', '.jpeg')
    return sorted([f for f in os.listdir(class_dir) if f.lower().endswith(valid_ext)])

def copy_files(file_list, src_dir, dst_dir):
    os.makedirs(dst_dir, exist_ok=True)
    for fname in file_list:
        shutil.copy2(os.path.join(src_dir, fname), os.path.join(dst_dir, fname))

split_summary = {}

for class_name in CLASSES:
    class_raw_dir = os.path.join(RAW_DIR, class_name)
    files = list_image_files(class_raw_dir)

    train_files, test_files = train_test_split(
        files, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )

    train_dst = os.path.join(SPLIT_DIR, 'train', class_name)
    test_dst = os.path.join(SPLIT_DIR, 'test', class_name)

    copy_files(train_files, class_raw_dir, train_dst)
    copy_files(test_files, class_raw_dir, test_dst)

    split_summary[class_name] = {
        'train': len(train_files),
        'test': len(test_files)
    }

    print(f"Kelas '{class_name}': {len(train_files)} train, {len(test_files)} test "
          f"(dari total {len(files)} gambar mentah)")

print("\nSplit selesai. Struktur baru:")
print(f"  {SPLIT_DIR}train/<kelas>/  -> sumber untuk augmentasi (notebook 03)")
print(f"  {SPLIT_DIR}test/<kelas>/   -> TIDAK BOLEH disentuh augmentasi, murni untuk evaluasi akhir")

Kelas 'healthy': 80 train, 20 test (dari total 100 gambar mentah)
Kelas 'sick': 80 train, 20 test (dari total 100 gambar mentah)

Split selesai. Struktur baru:
  ../data/split/train/<kelas>/  -> sumber untuk augmentasi (notebook 03)
  ../data/split/test/<kelas>/   -> TIDAK BOLEH disentuh augmentasi, murni untuk evaluasi akhir


In [7]:
# Sel 4: Fungsi Load dan Preprocessing
def load_and_preprocess_data():
    X = []
    y = []
    
    for label_idx, class_name in enumerate(CLASSES):
        class_dir = os.path.join(RAW_DIR, class_name)
        for img_name in os.listdir(class_dir):
            img_path = os.path.join(class_dir, img_name)
            
            # Baca gambar dan konversi ke RGB (OpenCV menggunakan BGR secara default)
            img = cv2.imread(img_path)
            if img is None:
                continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            # Resize ke 224x224 piksel
            img_resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            
            # Normalisasi piksel ke rentang [0,1] sesuai jurnal
            img_normalized = img_resized / 255.0
            
            X.append(img_normalized)
            y.append(label_idx)
            
    return np.array(X), np.array(y)

In [8]:
# Sel 5: Eksekusi Preprocessing
print("Memproses gambar...")
X_data, y_data = load_and_preprocess_data()
print(f"Total gambar diproses: {X_data.shape[0]}")
print(f"Dimensi data: {X_data.shape}")

Memproses gambar...
Total gambar diproses: 200
Dimensi data: (200, 224, 224, 3)


In [9]:
# Sel 6: Simpan Data yang Sudah Diproses
# Pastikan folder processed ada
os.makedirs(PROCESSED_DIR, exist_ok=True)
np.save(os.path.join(PROCESSED_DIR, 'X_data.npy'), X_data)
np.save(os.path.join(PROCESSED_DIR, 'y_data.npy'), y_data)
print("Data preprocessing selesai dan disimpan di data/processed/")

Data preprocessing selesai dan disimpan di data/processed/
